<a href="https://colab.research.google.com/github/zeynepkacar/skipedge-firmware-forensics/blob/main/pemu_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/MPI-SysSec/pemu.git


Cloning into 'pemu'...
remote: Enumerating objects: 312, done.
remote: Counting objects: 100% (312/312), done.
remote: Compressing objects: 100% (225/225), done.
remote: Total 312 (delta 72), reused 305 (delta 70), pack-reused 0 (from 0)
Receiving objects: 100% (312/312), 9.43 MiB | 17.79 MiB/s, done.
Resolving deltas: 100% (72/72), done.


In [ ]:
%cd pemu


/content/pemu


In [ ]:
!ls -la


total 32
drwxr-xr-x 7 root root 4096 Jul 31 20:13 .
drwxr-xr-x 1 root root 4096 Jul 31 20:13 ..
drwxr-xr-x 6 root root 4096 Jul 31 20:13 eval
drwxr-xr-x 8 root root 4096 Jul 31 20:13 .git
-rw-r--r-- 1 root root 2393 Jul 31 20:13 README.md
drwxr-xr-x 5 root root 4096 Jul 31 20:13 rehosting_platforms
drwxr-xr-x 3 root root 4096 Jul 31 20:13 scripts
drwxr-xr-x 4 root root 4096 Jul 31 20:13 src


In [ ]:
!cat README.md

# pemu-ae
## Structure
- `src/`: Contains the standalone source code of PEMU
- `scripts/`: Contains useful scripts for the evaluation
- `rehosting_platforms/`: Contains the evaluated rehosting platforms, and patches to prepare them for the use with PEMU
- `eval/`: Contains the sources and targets for the evaluation

## Setup
First, run `git submodule update --init --recursive`.   
Next, go to `eval/01-coverage-experiments/nrf52840/nuttx_nimble` and run `unzip nuttx.zip` (One of the binaries is too large for github).


Our tool serves as an add-on for existing rehosting platforms. For this paper we attached PEMU to three platforms: SEmu, Fuzzware, and Hoedur.  
This section describes how to setup everything for further fuzzing.  
After setting up a rehosting platform, all attached data and scripts can be found in `rehosting_platforms/<platform>`

### SEmu
To setup SEmu for fuzzing execute the script `/scripts/setup_semu.sh`. This script sets up two versions of SEmu, one with PEMU and on

In [ ]:
!ls -la src/

total 220
drwxr-xr-x 4 root root  4096 Jul 31 20:13 .
drwxr-xr-x 7 root root  4096 Jul 31 20:13 ..
drwxr-xr-x 2 root root  4096 Jul 31 20:13 analysis
-rw-r--r-- 1 root root 19553 Jul 31 20:13 artificial_network_interface.py
-rw-r--r-- 1 root root  9470 Jul 31 20:13 checksum.py
-rw-r--r-- 1 root root 38640 Jul 31 20:13 handler.py
-rw-r--r-- 1 root root 50112 Jul 31 20:13 packer.py
-rw-r--r-- 1 root root 23946 Jul 31 20:13 packet_parser.py
-rw-r--r-- 1 root root 27466 Jul 31 20:13 parsing_handler.py
-rw-r--r-- 1 root root   966 Jul 31 20:13 plugin_api.py
drwxr-xr-x 6 root root  4096 Jul 31 20:13 protocol_lib
-rw-r--r-- 1 root root 20481 Jul 31 20:13 utils.py


In [ ]:
!find src -name "*.md" -o -name "README*"

src/protocol_lib/README_protocols.yml


In [ ]:
!cat src/protocol_lib/README_protocols.yml


# This file aims to show all the different configuration options for protocol definitions

# It is mandatory to include a meta section that contains
meta:
  # The ID as defined in the protocols.yml file
  id: 
  # The endianess, i.e., big or little endian
  endian: be/le
  # (opt) If the endianess of the individual fields needs to specified
  field_endian: le
  # (opt) handshake
  handshake: 2


# Next we always need to include a header section
header:

  # However if the protocol does not need/use a header it is possible to add a dummy header
  # with size 0
  dummy_header:
    type: fuzzed
    size: 0
    doc: Just a dummy header

  # In general, each header field NEEDS 
  header_prototype:
    # a type, which can take any of the following values
    type: fuzzed/data/handler/condition/conditional/nested/length/reference/checksum/pointer
    # and a size either in bitwise (bX) or in bytewise granularity
    size: bX/X
    # optionally you can add information on the field
    doc: ...

In [ ]:
!ls -la src/protocol_lib/


total 36
drwxr-xr-x 6 root root 4096 Jul 31 20:13 .
drwxr-xr-x 4 root root 4096 Jul 31 20:13 ..
drwxr-xr-x 2 root root 4096 Jul 31 20:13 6lowpan
drwxr-xr-x 7 root root 4096 Jul 31 20:13 ble
drwxr-xr-x 2 root root 4096 Jul 31 20:13 industrial
-rw-r--r-- 1 root root 3227 Jul 31 20:13 protocols.yml
-rw-r--r-- 1 root root 6616 Jul 31 20:13 README_protocols.yml
drwxr-xr-x 2 root root 4096 Jul 31 20:13 tcpip_stack


In [ ]:
!ls -la src/analysis/

total 64
drwxr-xr-x 2 root root  4096 Jul 31 20:13 .
drwxr-xr-x 4 root root  4096 Jul 31 20:13 ..
-rw-r--r-- 1 root root 17534 Jul 31 20:13 comm_interface.py
-rw-r--r-- 1 root root 21618 Jul 31 20:13 probe_based_detection.py
-rw-r--r-- 1 root root 10823 Jul 31 20:13 save.py


In [ ]:
!cat src/protocol_lib/protocols.yml


name_id_mapping:
  Modbus: 0x50001
  Ethernet: 420
  TCP: 6
  UDP: 17
  IP4: 0x0800
  ARP: 0x0806
  IP6: 0x86dd
  ICMP6: 0x3a
  DHCP: 68
  DUP_DHCP: 67
  HTTP: 80
  SNMP: 0xa1
  COAP: 5683
  ICMP4: 1
  IGMP: 2
  IEEE802154_PHY: 0xfff
  6lowpan_MAC_Beacon: 0x20000
  6lowpan_MAC_Data: 0x20001
  6lowpan_MAC_Command: 0x20003
  6lowpan_MAC_Ack: 0x20002
  6lowpan: 0x20005
  6lowpan_IP6: 0x286dd
  6lowpan_mesh_frag: 0x20006

  # Remember ids are & 0xffff for pointer values
  BLE_PHY: 0xffff
  # Advertising and data channel
  BLE_adv_channel_pdu: 0x10009 # --> used id is 0x109
  BLE_data_channel_pdu: 0x10080 # --> used id is 0x180

  # advertisement pdus
  # legacy advertisements
  BLE_adv_ADV_IND: 0x10000 # don't change
  BLE_adv_ADV_DIRECT_IND: 0x10001 # don't change
  BLE_adv_ADV_NONCONN_IND: 0x10002 # don't change
  BLE_adv_ADV_SCAN_IND: 0x10006 # don't change
  # extended advertisements; they all have the same id 0b0111; hence this workaround
  BLE_adv_ADV_EXT_IND: 0x10007
  BLE_adv_AUX_A

In [ ]:
!grep -n "^class\|^def " src/packer.py | head -30


10:class Packer:


In [ ]:
!grep -n "^class\|^def " src/analysis/comm_interface.py | head -30

4:class CommunicatorInterface():
123:class FuzzwareCommunicator(CommunicatorInterface):
451:def parse_dma_file(path: str) -> dict:


In [ ]:
!sed -n '1,80p' src/packer.py

from .utils import *

place_data = be_place_data
NOT_ENOUGH_FUZZ = 0x12348765
OK = 1
EPHEMERAL_ATTR_NAMES = [
    "_waiting_fragments", "_handler_queue", "_snapshot", "_logger", "_consumed_fuzz", "_dirty", "_protocols", "_apriori_knowledge"
]

class Packer:
    """ Packer module which is responsible for encapsulating packets and storing the data across a run """
    _protocols: dict
    _static_protocols: dict
    _logger: logging
    _get_bytes: types.FunctionType
    _handler_queue: list
    _waiting_fragments: dict
    _apriori_knowledge: dict
    _static_apriori_knowledge: dict
    _faulty_layer: str
    _fault_probability: int
    _snapshot = dict()
    _ran_out_of_fuzz = bool
    _consumed_fuzz = list()
    _fuzz_consumed_per_packet: int
    _dirty: bool

    def __init__(self, config: dict, protocol_state: list, get_bytes: types.FunctionType, log_level=logging.WARNING):
        logging.basicConfig(level=logging.WARNING, format='%(message)s')
        self._logger = logging.getLog

In [ ]:
!grep -rn "Packer(" --include="*.py" . | grep -v "class Packer"

./src/artificial_network_interface.py:100:        self._packer = Packer(config, self._protocol_state, self._get_bytes, log_level)


In [ ]:
!sed -n '1,110p' src/artificial_network_interface.py

# TODO imports
from .utils import *
from .packer import Packer
from .packet_parser import PacketParser
from .plugin_api import APIGadget
import copy
import uuid

# Analysis might fail if the destination addresses are left empty.
# For this case we use addresses that can be interpreted as multi or broadcasts by the receiver
default_analysis_values = {'TCPIP.MAC_dest': 0xffffffffffff, 'TCPIP.IP4_dest': 0xffffffff}
EPHEMERAL_ATTR_NAMES = [
    "_tree_init", "_logger", "_snapshot", "_all_packets", "_network_tree", "_protocol_state", "_probe_mode", "_current_reception_state"
]
SNAPSHOTTABLE_ATTR_NAMES = [
    "_packer", "_parser"
]


class ANI:
    """
    This class represents the the entire network from the emulators perspective. It is responsible for
    providing packets, which can be passed to the firmware through a peripheral. It also parses packets
    the firmware is transmitting, to gain knowledge on context specific values.
    """
    _packer: Packer
    _parser: PacketParser
   

In [ ]:
!find . -iname "*.yml" -path "*eval*" | head -20
!find . -iname "*config*" -not -path "*/protocol_lib/*" | grep -v ".git" | head -20

./eval/01-coverage-experiments/stm32_f429/LwIP_TCP_Echo_Client/config_np.yml
./eval/01-coverage-experiments/stm32_f429/LwIP_TCP_Echo_Client/hoedur_config.yml
./eval/01-coverage-experiments/stm32_f429/LwIP_TCP_Echo_Client/config.yml
./eval/01-coverage-experiments/stm32_f429/LwIP_TCP_Echo_Client/config_dma.yml
./eval/01-coverage-experiments/stm32_f429/LwIP_TCP_Echo_Client/config_autogen.yml
./eval/01-coverage-experiments/stm32_f429/LwIP_TCP_Echo_Client/config_custom.yml
./eval/01-coverage-experiments/stm32_f429/LwIP_TCP_Echo_Server/config_np.yml
./eval/01-coverage-experiments/stm32_f429/LwIP_TCP_Echo_Server/hoedur_config.yml
./eval/01-coverage-experiments/stm32_f429/LwIP_TCP_Echo_Server/config.yml
./eval/01-coverage-experiments/stm32_f429/LwIP_TCP_Echo_Server/config_dma.yml
./eval/01-coverage-experiments/stm32_f429/LwIP_TCP_Echo_Server/config_autogen.yml
./eval/01-coverage-experiments/stm32_f429/LwIP_TCP_Echo_Server/config_custom.yml
./eval/01-coverage-experiments/ti_cc2538/snmp_server_c

In [ ]:
!cat eval/01-coverage-experiments/stm32_f429/LwIP_TCP_Echo_Client/config.yml

include:
  - config_autogen.yml
  - config_custom.yml


In [ ]:
!cat eval/01-coverage-experiments/stm32_f429/LwIP_TCP_Echo_Client/config_autogen.yml

interrupt_triggers:
  trigger:
    every_nth_tick: 0x3e8
    fuzz_mode: round_robin
memory_map:
  irq_ret:
    base_addr: 0xfffff000
    permissions: --x
    size: 0x1000
  mmio:
    base_addr: 0x40000000
    permissions: rw-
    size: 0x20000000
  nvic:
    base_addr: 0xe0000000
    permissions: rw-
    size: 0x10000000
  ram:
    base_addr: 0x20000000
    permissions: rw-
    size: 0x100000
  text:
    base_addr: 0x8000000
    file: LwIP_TCP_Echo_Client.bin
    ivt_offset: 0x0
    permissions: r-x
    size: 0x24124
symbols:
  0x0: reent.c
  0x200: _Min_Heap_Size
  0x400: _Min_Stack_Size
  0x8000000: g_pfnVectors
  0x80001b1: __do_global_dtors_aux
  0x80001d5: frame_dummy
  0x80001f1: strlen
  0x8000201: memchr
  0x80002a1: __aeabi_uldivmod
  0x80002d1: __udivmoddi4
  0x80005a1: __aeabi_idiv0
  0x80005a5: Reset_Handler
  0x80005ac: CopyDataInit
  0x80005b4: LoopCopyDataInit
  0x80005c2: FillZerobss
  0x80005c8: LoopFillZerobss
  0x80005f4: Default_Handler
  0x80005f5: DMA1_Stream3_IRQ

In [ ]:
!head -100 eval/01-coverage-experiments/stm32_f429/LwIP_TCP_Echo_Client/config_autogen.yml

interrupt_triggers:
  trigger:
    every_nth_tick: 0x3e8
    fuzz_mode: round_robin
memory_map:
  irq_ret:
    base_addr: 0xfffff000
    permissions: --x
    size: 0x1000
  mmio:
    base_addr: 0x40000000
    permissions: rw-
    size: 0x20000000
  nvic:
    base_addr: 0xe0000000
    permissions: rw-
    size: 0x10000000
  ram:
    base_addr: 0x20000000
    permissions: rw-
    size: 0x100000
  text:
    base_addr: 0x8000000
    file: LwIP_TCP_Echo_Client.bin
    ivt_offset: 0x0
    permissions: r-x
    size: 0x24124
symbols:
  0x0: reent.c
  0x200: _Min_Heap_Size
  0x400: _Min_Stack_Size
  0x8000000: g_pfnVectors
  0x80001b1: __do_global_dtors_aux
  0x80001d5: frame_dummy
  0x80001f1: strlen
  0x8000201: memchr
  0x80002a1: __aeabi_uldivmod
  0x80002d1: __udivmoddi4
  0x80005a1: __aeabi_idiv0
  0x80005a5: Reset_Handler
  0x80005ac: CopyDataInit
  0x80005b4: LoopCopyDataInit
  0x80005c2: FillZerobss
  0x80005c8: LoopFillZerobss
  0x80005f4: Default_Handler
  0x80005f5: DMA1_Stream3_IRQ

In [ ]:
!cat eval/01-coverage-experiments/stm32_f429/LwIP_TCP_Echo_Client/config_custom.yml

exit_at:
  DMA1_Stream3_IRQHandler: 0x80005f4

handlers:
  HAL_Delay_skip:
    addr: HAL_Delay
  LCD_Draw_char_skip:
    addr: BSP_LCD_DisplayChar
  puts_skip:
    addr: puts
  DHCP_periodic_handle_timer_skip:
    # Overwrite compare instruction to increase the frequency
    addr: 0x080007de
    handler: native.inline_asm_B0F1100F


nvic:
  disabled_irqs:
  # 48 - 16
  - 32
  - 56
  - 31  

memory_map:
  fsmc_region:
    # for interfacing with (a)synchronous memories
    base_addr: 0xa0000000
    size: 0x1000
    permissions: rw-
  bank1:
    base_addr: 0x60000000
    size: 0x10000000
    permissions: rw-
  bank2:
    base_addr: 0x70000000
    size: 0x10000000
    permissions: rw-
  bank3:
    base_addr: 0x80000000
    size: 0x10000000
    permissions: rw-
  bank4:
    base_addr: 0x90000000
    size: 0x10000000
    permissions: rw-
  sdram_bank1:
    base_addr: 0xc0000000
    size: 0x10000000
    permissions: rw-
  sdram_bank2:
    base_addr: 0xd0000000
    size: 0x10000000
    permiss

In [ ]:
!grep -n "protocol_state\|apriori\|protocols:" eval/01-coverage-experiments/stm32_f429/LwIP_TCP_Echo_Client/config_custom.yml eval/01-coverage-experiments/stm32_f429/LwIP_TCP_Echo_Client/config_autogen.yml

In [ ]:
!cat eval/01-coverage-experiments/stm32_f429/LwIP_TCP_Echo_Client/hoedur_config.yml

board:
  init_nsvtor: 0x8000000

interrupt:
  mode: round-robin
  trigger:
    on-infinite-sleep: true
    interval: 1000
  blocklist:
    - 48
    - 72
    - 47

network: 
  ani: pemu.artificial_network_interface.ANI
  np_only: true
  output: f429_tcp_client_output.txt
  faults:
    probability: 0x14
    target: all
  apriori:
    - field: DHCP.chaddr
      val: 0x2010000000000000000000000000000
    - field: DHCP.xid
      val: 0x4bb5f646
    - field: TCPIP.MAC_dest
      val:  0x20100000000
  protocols:
    - protocol: [Ethernet, IP4, UDP, DHCP]
    - protocol: [Ethernet, IP4, UDP, DHCP]
    - protocol: [Ethernet, IP4, TCP]
    - protocol: [Ethernet, ARP]
    - protocol: [Ethernet, IP4, TCP]

pemu_hooks:
  - address: 0x80008e2 # ethernetif_input
    hook_type: receive
    mcu: f7xx
  - address: 0x80031c2 # HAL_ETH_Transmit
    hook_type: transmit
    mcu: f7xx
  #  - address: 0x8006a70
  #    hook_type: crash
  #    mcu: f7xx
  #- address: 0x8000f98 
  #- address: 0x800afb8
memory_ma

In [ ]:
!grep -n "^def \|zero_rand\|get_bytes" src/utils.py | head -20

156:def zero_rand(n: int) -> bytes:
161:def be_place_data(data: int, bit_len: int):
166:def le_place_data(data: int, bit_len: int):
189:def resolve_checksum(field: dict, handler_key: str, store=True)-> types.FunctionType:
196:def resolve_handler(field: dict, handler_key: str, store=True, module=HANDLER_LOC) -> types.FunctionType:
221:def zero_bytes(*any):
228:def protocol_id_by_name(protocol: str) -> int:
241:def resolve_protocol_id(protocol: str or int):
261:def import_parsing_handler(name):
288:def add_handler(name, handler):
293:def get_filename(protocol: str):
301:def load_protocol(protocol: str or int):
314:def load_output_file(filename: str) -> dict:
323:def save_to_output_file(contents: dict, filename: str, force_save=False):
335:def get_field_info(field: dict, entry: str):
339:def set_field_info(field: dict, entry: str, data):
344:def set_header_field_size(layer: dict, name: str, data):
353:def set_field_size(field: dict, layer: dict, data):
366:def set_header_field(layer: dict

In [ ]:
import sys, os
sys.path.insert(0, '/content/pemu')
os.chdir('/content/pemu')

import yaml
from src.packer import Packer

def get_bytes(n):
    return os.urandom(n)

config = {
    'apriori': {
        'DHCP.chaddr': 0x2010000000000000000000000000000,
        'DHCP.xid': 0x4bb5f646,
        'TCPIP.MAC_dest': 0x20100000000,
    },
    'protocols': [
        {'protocol': ['Ethernet', 'ARP']},
    ],
}
protocol_state = [{'protocol': ['Ethernet', 'ARP']}]

packer = Packer(config, protocol_state, get_bytes)
print("Packer başarıyla oluşturuldu:", packer)

KeyError: 'protocol'

In [ ]:
protocol_state = ['Ethernet', 'ARP']  # dict sarmalayıcı olmadan, düz liste

packer = Packer(config, protocol_state, get_bytes)
print("Packer başarıyla oluşturuldu:", packer)

KeyError: 'E'

In [ ]:
!grep -n "_parse_config" -A 30 src/packer.py | head -50

39:        self._parse_config(config, protocol_state)
40-        self._static_protocols = copy.deepcopy(self._protocols)
41-        self._static_apriori_knowledge = copy.deepcopy(self._apriori_knowledge)
42-        self._dirty = False
43-
44-
45-    def take_snapshot(self):
46-        """ We need to maintain our own snapshot. As the ANI is not able to store complex objects (like Packer) """
47-        for attr_name, value in self.__dict__.items():
48-            if attr_name in EPHEMERAL_ATTR_NAMES:
49-                continue
50-
51-            self._snapshot[attr_name] = getattr(self, copy.deepcopy(attr_name))
52-        return
53-
54-    
55-    def restore_snapshot(self):
56-        """ Restore the saved snapshot; Use a dirty flag to avoid unnecessary restores """
57-        if not self._dirty:
58-            # If nothing was changed we don't need to restore anything
59-            return
60-        self._dirty = False
61-
62-        for attr_name, val in self._snapshot.items():
63

In [ ]:
protocol_state = [['Ethernet', 'ARP']]  # liste içinde liste

packer = Packer(config, protocol_state, get_bytes)
print("Packer başarıyla oluşturuldu:", packer)

Packer başarıyla oluşturuldu: <src.packer.Packer object at 0x782d184ac530>


In [ ]:
!grep -n "^    def " src/packer.py

28:    def __init__(self, config: dict, protocol_state: list, get_bytes: types.FunctionType, log_level=logging.WARNING):
45:    def take_snapshot(self):
55:    def restore_snapshot(self):
73:    def discard_snapshot(self):
77:    def _switch_to_zero_randomness(self):
83:    def _get_fuzz(self, n: int) -> bytes:
104:    def _parse_config(self, config: dict, protocol_seq: list):
168:    def _choose_fault_layer(self, pkt: list):
183:    def has_fragments_waiting(self) -> bool:
191:    def _is_handshake(self, packet: list) -> bool:
213:    def _import_configs(self, packet: list):
221:    def get_packet(self, packet: list) -> (bytes, bool):
295:    def _switch_endianess(self, layer: dict):
311:    def _assemble_body(self, layer: str):
366:    def _parse_payload(self, layer: dict, upper_layer: str):
380:    def _assemble_header(self, layer: dict, packet: bytes, upper_layer: str):
494:    def _sort_fields(self, name: str, field: dict, grouped: dict):
506:    def _perform_actions(self, grouped

In [ ]:
pkt_bytes, success = packer.get_packet(['Ethernet', 'ARP'])
print("Başarılı mı:", success)
print("Üretilen paket (hex):", pkt_bytes.hex())
print("Paket boyutu:", len(pkt_bytes), "byte")

ValueError: too many values to unpack (expected 2)

In [ ]:
!sed -n '221,295p' src/packer.py

    def get_packet(self, packet: list) -> (bytes, bool):
        """
        Assemble a packet from a list representation

        Args:
            packet: List representation of the packet

        Returns:
            Binary data, which represents the assembled packet
        """

        global place_data
        cur_pkt = list(packet)

        if self._ran_out_of_fuzz:
            # If the last try to get a packet ran out of fuzz, reset and try again
            self._consumed_fuzz = self._consumed_fuzz[:(len(self._consumed_fuzz) - self._fuzz_consumed_per_packet)]
            self._ran_out_of_fuzz = False

        self._fuzz_consumed_per_packet = 0
        self._dirty = True

        self._import_configs(packet)
        handshake =  self._is_handshake(cur_pkt)

        # If there is a handshake we only encapsulate up to the handshaking layer until the handshake is done
        if handshake:
            self._protocols[cur_pkt[-1]][BODY][SIZE] = 0
            previous_layer = packe

In [ ]:
pkt_bytes = packer.get_packet(['Ethernet', 'ARP'])
print("Üretilen paket (hex):", pkt_bytes.hex())
print("Paket boyutu:", len(pkt_bytes), "byte")

Üretilen paket (hex): ffffffffffff941a122e26f70806000186dd06040002941a122e26f73651fff4ffffffffffff32a42749
Paket boyutu: 42 byte
